# Final Repository Assembly From Google Drive

Run this after the authoritative executed notebook has completed. It validates the Drive repository, allocates MS2 files and creates a release ZIP.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
REPO_ROOT = Path('/content/drive/MyDrive/cross-ancestry-asthma-transcriptomics')
print('Repository:', REPO_ROOT)
assert REPO_ROOT.exists(), 'Repository folder not found'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository: /content/drive/MyDrive/cross-ancestry-asthma-transcriptomics


In [ ]:
import pandas as pd
manifest = REPO_ROOT/'results/manuscript_1/MANIFEST.tsv'
assert manifest.exists(), manifest
m = pd.read_csv(manifest, sep='\t')
print('Files indexed:', len(m))
assert len(m) == 72, f'Expected 72, found {len(m)}'
display(m.groupby('section').size().rename('files'))


Files indexed: 72


,files
section,
main_figures,16
main_figures/pathway_violins,10
main_tables,6
supplementary/S1_complete_DE_and_expression,9
supplementary/S2_pathway_and_enrichment,4
supplementary/S2_pathway_and_enrichment/MS1_GSEA_ancestry_adjusted,3
supplementary/S3_curated_gene_statistics,3
supplementary/S4_coexpression_networks,21


In [ ]:
# Validate the key result counts directly from the generated tables.
de = pd.read_csv(REPO_ROOT/'results/manuscript_1/supplementary/S1_complete_DE_and_expression/MS1_S1A_primary_ancestry_adjusted_OLS.tsv.gz', sep='\t')
assert int((de.fdr < 0.05).sum()) == 2134
assert int(((de.fdr < 0.05) & (de.mean_difference.abs() > 0.5)).sum()) == 1020
assert int(((de.fdr < 0.05) & (de.mean_difference > 0.5)).sum()) == 612
assert int(((de.fdr < 0.05) & (de.mean_difference < -0.5)).sum()) == 408
sex = pd.read_csv(REPO_ROOT/'results/manuscript_1/supplementary/S1_complete_DE_and_expression/MS1_S1C_primary_sex_adjusted_OLS.tsv.gz', sep='\t')
assert int((sex.fdr < 0.05).sum()) == 97
assert int(((sex.fdr < 0.05) & (sex.mean_difference.abs() > 0.5)).sum()) == 73
print('Key differential-expression checks passed.')


Key differential-expression checks passed.


In [ ]:
import numpy as np
import pandas as pd

print("=" * 70)
print("PHASE E — GSEA AND NETWORK VERIFICATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. GSEA
# ------------------------------------------------------------
gsea_path = (
    REPO_ROOT /
    'results/manuscript_1/supplementary/'
    'S2_pathway_and_enrichment/'
    'MS1_S2A_GSEA_ancestry_adjusted.tsv'
)

assert gsea_path.exists(), f"Missing GSEA table: {gsea_path}"

gsea = pd.read_csv(gsea_path, sep='\t')

term_col = next(
    c for c in ['Term', 'pathway', 'Pathway']
    if c in gsea.columns
)
fdr_col = next(
    c for c in ['FDR q-val', 'fdr', 'FDR']
    if c in gsea.columns
)
nes_col = next(
    c for c in ['NES', 'nes']
    if c in gsea.columns
)

def norm_name(x):
    return (
        str(x).strip().lower()
        .replace(' ', '_')
        .replace('-', '_')
    )

gsea['_norm'] = gsea[term_col].map(norm_name)

epi = gsea.loc[
    gsea['_norm'].isin(['epithelial_barrier_function'])
]

air = gsea.loc[
    gsea['_norm'].isin(
        ['airway_remodelling', 'airway_remodeling']
    )
]

assert len(epi) == 1, \
    "Could not uniquely identify Epithelial Barrier Function"
assert len(air) == 1, \
    "Could not uniquely identify Airway Remodelling"

epi_fdr = float(epi.iloc[0][fdr_col])
air_fdr = float(air.iloc[0][fdr_col])
epi_nes = float(epi.iloc[0][nes_col])
air_nes = float(air.iloc[0][nes_col])

print()
print(
    "Epithelial Barrier Function:",
    f"NES = {epi_nes:.4f}",
    f"FDR = {epi_fdr:.4f}"
)

print(
    "Airway Remodelling:",
    f"NES = {air_nes:.4f}",
    f"FDR = {air_fdr:.4f}"
)

assert np.isclose(epi_fdr, 0.0152, atol=0.00005), \
    f"Unexpected Epithelial Barrier FDR: {epi_fdr}"

assert np.isclose(air_fdr, 0.0193, atol=0.00005), \
    f"Unexpected Airway Remodelling FDR: {air_fdr}"

# ------------------------------------------------------------
# 2. Full co-expression network
# ------------------------------------------------------------
network_path = (
    REPO_ROOT /
    'results/manuscript_1/supplementary/'
    'S4_coexpression_networks/'
    'MS1_S4_network_summary.tsv'
)

assert network_path.exists(), \
    f"Missing network summary: {network_path}"

network = pd.read_csv(network_path, sep='\t')

row = network.loc[
    network['group']
    .astype(str)
    .str.strip()
    .str.lower()
    .eq('all samples')
]

assert len(row) == 1, \
    "Could not uniquely identify the All samples network"

tested_pairs = int(row.iloc[0]['tested_pairs'])
edges = int(row.iloc[0]['retained_edges'])
connected_nodes = int(row.iloc[0]['connected_nodes'])

print()
print("Full network tested pairs:", tested_pairs)
print("Full network retained edges:", edges)
print("Full network connected nodes:", connected_nodes)

assert tested_pairs == 8128, \
    f"Expected 8128 tested pairs, found {tested_pairs}"

assert edges == 173, \
    f"Expected 173 retained edges, found {edges}"

assert connected_nodes == 81, \
    f"Expected 81 connected nodes, found {connected_nodes}"

print()
print("GSEA and full-network verification PASSED.")

PHASE E — GSEA AND NETWORK VERIFICATION

Epithelial Barrier Function: NES = -1.8710 FDR = 0.0152
Airway Remodelling: NES = -1.7659 FDR = 0.0193

Full network tested pairs: 8128
Full network retained edges: 173
Full network connected nodes: 81

GSEA and full-network verification PASSED.


In [ ]:
# Allocate shared outputs to manuscript_2 without modifying manuscript_1.
import shutil
MS1 = REPO_ROOT/'results/manuscript_1'
MS2 = REPO_ROOT/'results/manuscript_2'
for d in [MS2/'main_tables', MS2/'main_figures', MS2/'supplementary']:
    d.mkdir(parents=True, exist_ok=True)
copy_map = {
    MS1/'supplementary/S2_pathway_and_enrichment/MS1_S2A_GSEA_ancestry_adjusted.tsv': MS2/'main_tables/MS2_Table_1_GSEA.tsv',
    MS1/'supplementary/S2_pathway_and_enrichment/MS1_S2C_ssGSEA_ancestry_adjusted_OLS.tsv': MS2/'main_tables/MS2_Table_2_ssGSEA.tsv',
    MS1/'supplementary/S1_complete_DE_and_expression/MS1_S1C_primary_sex_adjusted_OLS.tsv.gz': MS2/'supplementary/MS2_Table_S3_sex_probe_results.tsv.gz',
    MS1/'supplementary/S4_coexpression_networks/MS1_S4_network_summary.tsv': MS2/'main_tables/MS2_Table_4_network_summary.tsv',
    MS1/'supplementary/S4_coexpression_networks/MS1_S4_all_samples_nodes.tsv': MS2/'supplementary/MS2_Table_S4_nodes.tsv',
    MS1/'supplementary/S4_coexpression_networks/MS1_S4_all_samples_edges.tsv': MS2/'supplementary/MS2_Table_S4_edges.tsv',
}
for src, dst in copy_map.items():
    assert src.exists(), src
    shutil.copy2(src, dst)
    print('Copied', dst.relative_to(REPO_ROOT))


Copied results/manuscript_2/main_tables/MS2_Table_1_GSEA.tsv
Copied results/manuscript_2/main_tables/MS2_Table_2_ssGSEA.tsv
Copied results/manuscript_2/supplementary/MS2_Table_S3_sex_probe_results.tsv.gz
Copied results/manuscript_2/main_tables/MS2_Table_4_network_summary.tsv
Copied results/manuscript_2/supplementary/MS2_Table_S4_nodes.tsv
Copied results/manuscript_2/supplementary/MS2_Table_S4_edges.tsv


In [ ]:
import pandas as pd

print("=" * 70)
print("PHASE E — CURATED GENE TABLE VERIFICATION")
print("=" * 70)

curated_path = (
    REPO_ROOT /
    'results/manuscript_1/main_tables/'
    'MS1_Table_3_curated_genes_revised.tsv'
)

assert curated_path.exists(), \
    f"Missing curated-gene table: {curated_path}"

curated = pd.read_csv(curated_path, sep='\t')

print("Curated-gene rows:", len(curated))
print("Columns:", curated.columns.tolist())

assert len(curated) == 13, \
    f"Expected 13 rows, found {len(curated)}"

assert 'fdr' in curated.columns, \
    "Expected an 'fdr' column"

assert (curated['fdr'] < 0.05).all(), \
    "At least one curated gene has FDR >= 0.05"

gene_col = (
    'gene_symbol'
    if 'gene_symbol' in curated.columns
    else 'gene'
)

expected_genes = {
    'MMP9',
    'CXCL1',
    'CXCL8',
    'ALOX5',
    'ARRB2',
    'TIMP2',
    'TGFB1',
    'LTC4S',
    'CLDN18',
    'GATA3',
    'LDHA',
    'CD28',
    'PRDX1'
}

actual_genes = set(
    curated[gene_col].astype(str).str.strip()
)

assert actual_genes == expected_genes, \
    (
        f"Gene-set mismatch.\n"
        f"Missing: {expected_genes - actual_genes}\n"
        f"Unexpected: {actual_genes - expected_genes}"
    )

display(
    curated.sort_values('fdr')
)

print()
print("13-row curated-gene table structural verification PASSED.")

PHASE E — CURATED GENE TABLE VERIFICATION
Curated-gene rows: 13
Columns: ['gene_symbol', 'positive_level', 'reference_level', 'covariate', 'positive_mean', 'reference_mean', 'unadjusted_mean_difference', 'mean_difference', 'standard_error', 't_statistic', 'p_value', 'cohens_d', 'df_residual', 'test', 'fdr', 'significant', 'correction_family']


,gene_symbol,positive_level,reference_level,covariate,positive_mean,reference_mean,unadjusted_mean_difference,mean_difference,standard_error,t_statistic,p_value,cohens_d,df_residual,test,fdr,significant,correction_family
0,ALOX5,African,European,gender,7.088958,7.695000,-0.606042,-0.606042,0.097258,-6.231262,0.000004,-2.495233,21,OLS adjusted for gender,0.000450,True,BH across all 128 curated genes
1,ARRB2,African,European,gender,9.937500,10.389167,-0.451667,-0.451667,0.086849,-5.200605,0.000037,-2.096690,21,OLS adjusted for gender,0.001595,True,BH across all 128 curated genes
2,MMP9,African,European,gender,7.076667,8.809167,-1.732500,-1.732500,0.328489,-5.274154,0.000031,-1.998620,21,OLS adjusted for gender,0.001595,True,BH across all 128 curated genes
3,CD28,African,European,gender,5.165000,4.857500,0.307500,0.307500,0.063987,4.805639,0.000095,1.405187,21,OLS adjusted for gender,0.003039,True,BH across all 128 curated genes
4,TIMP2,African,European,gender,9.862778,10.233333,-0.370556,-0.370556,0.079394,-4.667283,0.000132,-1.782965,21,OLS adjusted for gender,0.003378,True,BH across all 128 curated genes
5,LDHA,African,European,gender,11.050833,10.698333,0.352500,0.352500,0.084573,4.168001,0.000435,1.737738,21,OLS adjusted for gender,0.008842,True,BH across all 128 curated genes
6,CXCL1,African,European,gender,6.299167,7.278333,-0.979167,-0.979167,0.240082,-4.078468,0.000539,-1.703893,21,OLS adjusted for gender,0.008842,True,BH across all 128 curated genes
7,CLDN18,African,European,gender,3.107083,3.255208,-0.148125,-0.148125,0.036415,-4.067741,0.000553,-1.498188,21,OLS adjusted for gender,0.008842,True,BH across all 128 curated genes
8,TGFB1,African,European,gender,6.167083,6.567917,-0.400833,-0.400833,0.101172,-3.961918,0.000712,-1.628460,21,OLS adjusted for gender,0.010120,True,BH across all 128 curated genes
9,PRDX1,African,European,gender,10.179167,9.886667,0.292500,0.292500,0.080650,3.626785,0.001580,1.511235,21,OLS adjusted for gender,0.020225,True,BH across all 128 curated genes



13-row curated-gene table structural verification PASSED.


In [ ]:
import pandas as pd

print("=" * 70)
print("PHASE E — NETWORK CENTRALITY / HUB AUDIT")
print("=" * 70)

nodes_path = (
    REPO_ROOT /
    'results/manuscript_1/supplementary/'
    'S4_coexpression_networks/'
    'MS1_S4_all_samples_nodes.tsv'
)

assert nodes_path.exists(), \
    f"Missing node table: {nodes_path}"

nodes = pd.read_csv(nodes_path, sep='\t')

print("Node-table rows:", len(nodes))
print("Columns:", nodes.columns.tolist())

required = {'gene', 'degree', 'betweenness'}

assert required.issubset(nodes.columns), \
    f"Required columns missing. Found: {nodes.columns.tolist()}"

connected = nodes.loc[nodes['degree'] > 0].copy()

connected['degree_rank'] = (
    connected['degree']
    .rank(method='min', ascending=False)
    .astype(int)
)

connected['betweenness_rank'] = (
    connected['betweenness']
    .rank(method='min', ascending=False)
    .astype(int)
)

display_cols = [
    c for c in [
        'gene',
        'pathway_membership',
        'degree',
        'degree_rank',
        'betweenness',
        'betweenness_rank'
    ]
    if c in connected.columns
]

print()
print("TOP 15 GENES BY DEGREE")
display(
    connected
    .sort_values(
        ['degree', 'betweenness'],
        ascending=[False, False]
    )
    .head(15)[display_cols]
)

print()
print("TOP 15 GENES BY BETWEENNESS")
display(
    connected
    .sort_values(
        ['betweenness', 'degree'],
        ascending=[False, False]
    )
    .head(15)[display_cols]
)

candidate_genes = [
    'GSTP1', 'GSTM1', 'GSTM3',
    'CYP3A4', 'CYP3A5', 'CYP2D6',
    'ALOX5', 'HMOX1', 'NR3C1',
    'ARRB2', 'NQO1'
]

print()
print("CANDIDATE GENES REFERENCED IN NETWORK INTERPRETATION")

display(
    connected.loc[
        connected['gene'].isin(candidate_genes)
    ]
    .sort_values(
        ['degree_rank', 'betweenness_rank']
    )[display_cols]
)

PHASE E — NETWORK CENTRALITY / HUB AUDIT
Node-table rows: 128
Columns: ['gene', 'pathway', 'degree', 'betweenness']

TOP 15 GENES BY DEGREE


,gene,degree,degree_rank,betweenness,betweenness_rank
1,COL1A1,13,1,0.033836,2
0,ARRB2,13,1,0.025917,9
2,SPDEF,12,3,0.047089,1
3,CD28,11,4,0.031973,3
4,LDHA,11,4,0.013694,21
5,ALOX5,10,6,0.021350,12
8,MMP9,9,7,0.029314,6
6,CXCL1,9,7,0.028486,7
9,TIMP2,9,7,0.014752,19
7,EDNRA,9,7,0.012405,23



TOP 15 GENES BY BETWEENNESS


,gene,degree,degree_rank,betweenness,betweenness_rank
2,SPDEF,12,3,0.047089,1
1,COL1A1,13,1,0.033836,2
3,CD28,11,4,0.031973,3
21,SMAD2,6,22,0.031649,4
40,SMAD4,3,39,0.029996,5
8,MMP9,9,7,0.029314,6
6,CXCL1,9,7,0.028486,7
19,TLR4,7,18,0.025938,8
0,ARRB2,13,1,0.025917,9
57,CYP2C9,2,45,0.022872,10



CANDIDATE GENES REFERENCED IN NETWORK INTERPRETATION


,gene,degree,degree_rank,betweenness,betweenness_rank
0,ARRB2,13,1,0.025917,9
5,ALOX5,10,6,0.021350,12
34,NR3C1,4,32,0.016908,16
38,GSTM1,3,39,0.005561,34
77,GSTM3,1,62,0.000000,54


In [ ]:
ranked_nodes = nodes.copy()

ranked_nodes['degree_rank'] = (
    ranked_nodes['degree']
    .rank(method='min', ascending=False)
    .astype(int)
)

ranked_nodes['betweenness_rank'] = (
    ranked_nodes['betweenness']
    .rank(method='min', ascending=False)
    .astype(int)
)

out = (
    REPO_ROOT /
    'results/manuscript_2/supplementary/'
    'MS2_Table_S4_nodes_ranked.tsv'
)

out.parent.mkdir(parents=True, exist_ok=True)

ranked_nodes.sort_values(
    ['degree_rank', 'betweenness_rank']
).to_csv(
    out,
    sep='\t',
    index=False
)

print("Saved:", out)
print("Rows:", len(ranked_nodes))

Saved: /content/drive/MyDrive/cross-ancestry-asthma-transcriptomics/results/manuscript_2/supplementary/MS2_Table_S4_nodes_ranked.tsv
Rows: 128


In [ ]:
from pathlib import Path

files_to_check = [
    REPO_ROOT /
    'results/manuscript_2/supplementary/'
    'MS2_Table_S4_nodes.tsv',

    REPO_ROOT /
    'results/manuscript_2/supplementary/'
    'MS2_Table_S4_edges.tsv',

    REPO_ROOT /
    'results/manuscript_2/supplementary/'
    'MS2_Table_S4_nodes_ranked.tsv',
]

for p in files_to_check:
    print(
        "PASS" if p.exists() else "MISSING",
        "—",
        p
    )

PASS — /content/drive/MyDrive/cross-ancestry-asthma-transcriptomics/results/manuscript_2/supplementary/MS2_Table_S4_nodes.tsv
PASS — /content/drive/MyDrive/cross-ancestry-asthma-transcriptomics/results/manuscript_2/supplementary/MS2_Table_S4_edges.tsv
PASS — /content/drive/MyDrive/cross-ancestry-asthma-transcriptomics/results/manuscript_2/supplementary/MS2_Table_S4_nodes_ranked.tsv


In [ ]:
# Build a checksum index and a repository ZIP for local GitHub transfer.
import hashlib, zipfile
def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()
rows=[]
for path in REPO_ROOT.rglob('*'):
    if path.is_file() and '.git' not in path.parts:
        rows.append({'relative_path':str(path.relative_to(REPO_ROOT)), 'size_bytes':path.stat().st_size, 'sha256':sha256(path)})
checksums=pd.DataFrame(rows).sort_values('relative_path')
checksums.to_csv(REPO_ROOT/'RELEASE_MANIFEST.tsv', sep='\t', index=False)
zip_path = REPO_ROOT.parent/'cross-ancestry-asthma-transcriptomics_ANALYSIS_SNAPSHOT.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for path in REPO_ROOT.rglob('*'):
        if path.is_file() and '.git' not in path.parts:
            zf.write(path, arcname=str(Path(REPO_ROOT.name)/path.relative_to(REPO_ROOT)))
print('Release ZIP:', zip_path)
print('Files:', len(checksums))


Release ZIP: /content/drive/MyDrive/cross-ancestry-asthma-transcriptomics_ANALYSIS_SNAPSHOT.zip
Files: 99


## Next
Download the release ZIP to your Windows computer, copy it into the local GitHub Desktop repository, run the local verification script, commit and push.